In [3]:
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
import pickle
import pandas as pd
import numpy as np
 

In [4]:
#Load the trained model, scaler pickle,onehot

model = load_model('model.h5')

# load the encoder and scaler

with open('onehot_encoder_geo.pkl','rb') as file:
    label_encoder_geo =pickle.load(file)

with open('label_encoder_gender.pkl','rb') as file:
    
    label_encoder_gender =pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler =pickle.load(file)

In [19]:
# 1. Define Synthetic Input Data (Raw Format)
input_data = {
    'CreditScore': 600, 
    'Geography': 'France',
    'Gender': 'Male', 
    'Age': 35,
    'Tenure': 5, 
    'Balance': 60000.0, 
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000.0
}

In [20]:
geo_encoded = label_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=label_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

c:\Users\Lenovo\Desktop\Ann Project\.learn\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [21]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,35,5,60000.0,2,1,1,50000.0


In [23]:
#Encode the categorical Data
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,35,5,60000.0,2,1,1,50000.0


In [24]:
#Concatination one hot encoded
input_df=pd.concat([input_df.drop('Geography',axis=1),geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,35,5,60000.0,2,1,1,50000.0,1.0,0.0,0.0


In [25]:
#scaling the data 
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755, -0.37056859, -0.00134472, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [26]:
#prediction churn
prediction=model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step


array([[0.00737293]], dtype=float32)

In [27]:
prediction_proba =prediction[0][0]

if prediction_proba> 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
